# Healthcare Urgency Classification Using Natural Language Processing



In [310]:
#Library import 

import pandas as pd
import string
import pandas as pd
!pip install nltk
import re
from nltk.corpus import stopwords
import nltk
from nltk.tokenize import word_tokenize
nltk.download('wordnet')
nltk.download('omw-1.4')
from nltk.stem import WordNetLemmatizer
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')


[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\razan\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\razan\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\razan\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\razan\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\razan\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [311]:
#import dataset 

df = pd.read_csv("dataset.csv", sep=';')
print(df.columns)
df.head()

Index(['text', 'label'], dtype='object')


,text,label
0,"severe abdominal pain huardign and rebound tenderness on exam. unclear onset,URGENT",URGENT
1,"laceartion to forearm requiring sutures bleeding controlled,URGENT",URGENT
2,"no acute distress here perop for evaluation surgery next week. difficult to determine acuity,ROUTINE",ROUTINE
3,"followup for stable eczema no new zomplaints. might be related to prior issue,NON-URGENT",NON-URGENT
4,"patient anaphylaxis airway swellign and hypotension ntoed. unclear onset. vitals stable, patient conversant,EMERGENCY",EMERGENCY


# First step- Data Preprocessing 

In [313]:
df["original_text"] = df["text"]

In [314]:
# Example Before Cleaning:
print("Before Cleaning:\n")
print(df["original_text"].iloc[6])


Before Cleaning:

high fever and vomiting persistent for days signs of uehydration,URGENT


In [315]:
#Converting Text to Lowercase:

df["text"] = df["text"].str.lower()
df["text"].iloc[4]

#Removing Punctuation from Text:

df["text"] = df["text"].apply(lambda x: x.translate(str.maketrans("", "", string.punctuation)))
df["text"].iloc[2]

#Text Cleaning: Lowercasing and Removing Punctuation:

df["cleaned"] =df["text"].str.lower().apply(lambda x: x.translate(str.maketrans("", "", string.punctuation)))

df[["text", "cleaned"]].head(3)


#Example After Cleaning:
print("\nAfter Cleaning:\n")
print(df["text"].iloc[6])

 


After Cleaning:

high fever and vomiting persistent for days signs of uehydrationurgent


In [316]:
# Example Before Tokens:
print("Example before tokns:")
print(df['text'].iloc[0])

Example before tokns:
severe abdominal pain huardign and rebound tenderness on exam unclear onseturgent


In [317]:
#Loading English Stopwords:

stop_words = set(stopwords.words('english'))

# Selecting a Text Sample from the Dataset
text = df['text'].iloc[0]

#Text Cleaning, Tokenization, and Stopword Removal:

text_clean = text.lower()
text_clean = re.sub(r'[^a-z\s]', '', text_clean)
tokens = [word for word in text_clean.split() if word not in stop_words]

#Example After Tokns:
print("Example after tokns:")
print(' '.join(tokens))

df.head()

Example after tokns:
severe abdominal pain huardign rebound tenderness exam unclear onseturgent


,text,label,original_text,cleaned
0,severe abdominal pain huardign and rebound tenderness on exam unclear onseturgent,URGENT,"severe abdominal pain huardign and rebound tenderness on exam. unclear onset,URGENT",severe abdominal pain huardign and rebound tenderness on exam unclear onseturgent
1,laceartion to forearm requiring sutures bleeding controlledurgent,URGENT,"laceartion to forearm requiring sutures bleeding controlled,URGENT",laceartion to forearm requiring sutures bleeding controlledurgent
2,no acute distress here perop for evaluation surgery next week difficult to determine acuityroutine,ROUTINE,"no acute distress here perop for evaluation surgery next week. difficult to determine acuity,ROUTINE",no acute distress here perop for evaluation surgery next week difficult to determine acuityroutine
3,followup for stable eczema no new zomplaints might be related to prior issuenonurgent,NON-URGENT,"followup for stable eczema no new zomplaints. might be related to prior issue,NON-URGENT",followup for stable eczema no new zomplaints might be related to prior issuenonurgent
4,patient anaphylaxis airway swellign and hypotension ntoed unclear onset vitals stable patient conversantemergency,EMERGENCY,"patient anaphylaxis airway swellign and hypotension ntoed. unclear onset. vitals stable, patient conversant,EMERGENCY",patient anaphylaxis airway swellign and hypotension ntoed unclear onset vitals stable patient conversantemergency


In [318]:
#Example Before Stopwords Removal:

print("Before Stopwords Removal:\n")
print(df["text"].iloc[0])

Before Stopwords Removal:

severe abdominal pain huardign and rebound tenderness on exam unclear onseturgent


In [319]:
#Removing Stopwords

def remove_stopwords(text):

    words = word_tokenize(text)

    filtered_words = [word for word in words if word.lower() not in stop_words]

    return " ".join(filtered_words)

df.columns

df["clean_text"] = df["text"].apply(remove_stopwords)

#Example After Stopwords Removal:
 
print("\nAfter Stopwords Removal:\n")
print(df["clean_text"].iloc[0])


After Stopwords Removal:

severe abdominal pain huardign rebound tenderness exam unclear onseturgent


In [320]:
#Example Before Lemmatization:

print("Before Lemmatization:\n")
print(df["clean_text"].iloc[6])


Before Lemmatization:

high fever vomiting persistent days signs uehydrationurgent


In [323]:
#Applying Lemmatization to Normalize Words:

lemmatizer = WordNetLemmatizer()

def lemmatize_text(text):
    
    words = word_tokenize(text)
    
    lemmas = [lemmatizer.lemmatize(word) for word in words]
    
    return " ".join(lemmas)

df["lemmatized_text"] = df["clean_text"].apply(lemmatize_text)

#Example After Lemmatization:

print("\nAfter Lemmatization:\n")
print(df["lemmatized_text"].iloc[6])


After Lemmatization:

high fever vomiting persistent day sign uehydrationurgent


# Second step- Feature Extraction

In [334]:
#Feature Extraction using TF-IDF

import pandas as pd

data = pd.read_csv("dataset.csv", sep=';')

texts = df["lemmatized_text"]

from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=1000, ngram_range=(1,2))

X = vectorizer.fit_transform(texts)

print("Shape of TF-IDF Matrix:")
print(X.shape)

Shape of TF-IDF Matrix:
(500, 1000)


In [336]:
#Feature Inspection 

print("Number of features:", len(vectorizer.get_feature_names_out()))
print("Sample features:", vectorizer.get_feature_names_out()[:10])

Number of features: 1000
Sample features: ['abdomen' 'abdomen active' 'abdomen atcive' 'abdominal' 'abdominal pain'
 'abnormal' 'abnormal denies' 'abnormal difficult' 'abnormal history'
 'abnormal might']


In [338]:
#Example 
print("Original Text:")
print(texts.iloc[0])
print("\nTF-IDF Representation:")
print(X[0].toarray())

Original Text:
severe abdominal pain huardign rebound tenderness exam unclear onseturgent

TF-IDF Representation:
[[0.         0.         0.         0.27397325 0.28452997 0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         